# Byte Pair Encoding (BPE) Tokenizer From Scratch

## 1. The main idea behind byte pair encoding (BPE)

- LLMs cannot work directly with categorical data like text - you cannot perform mathematical operations on text
- The main idea in BPE is to convert text into an integer representation (token IDs) for LLM training

### 1.1 Bits and Bytes

- Before we dive into BPE algorithm, let's introduce the notion of bytes (BPE stands for **byte** pair encoding)
- Consider converting text into a byte array

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- When we call `list()` on a `bytearray` object, each byte is treated as an individual element, and the result is a list of integers corresponding to the byte values:

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- First intuition would be to assume that this is a valid way to convert text into a token ID representation that we need for embedding layer of an LLM
- However, the downside of this approach is that it is creating one ID for each character - this is a lot of IDs for a short text like this
- I.e., this means that for a 17-character input text, we have to use 17 token IDs as input to the LLM:

In [3]:
print(f"Number of characters: {len(text)}")
print(f"Number of token IDs: {len(ids)}")

Number of characters: 17
Number of token IDs: 17


- We know that the BPE tokenizers have a vocabulary where a token ID can represent whole words or subwords instead of each character
- For example, the GPT-2 tokenizer tokenizes the same text ("This is some text") into 4 instead of 17 tokens: `1212, 318, 617, 2420`
- You can double check this using the interactive [tiktoken app](https://platform.openai.com/tokenizer) or using the tiktoken library

In [4]:
import tiktoken

In [5]:
gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")

[1212, 318, 617, 2420]

- Since a byte consists of 8 bits, there are $2^8=256$ possible values that a single byte can represent, ranging from 0 to 255
- You can confirm this by executing the code `bytearray(range(0, 257))`, which will warn you that `ValueError: byte must be in range(0, 256)`

In [6]:
bytearray(range(0, 257))

ValueError: byte must be in range(0, 256)

- A BPE tokenizer usually uses these 256 values as its first 256 single-character tokens. We can visually check this by running the following code:

In [ ]:
for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i:<8} | {decoded:^8} | {len(decoded):>8}")
"""
0    |  !   |    1
1    |  "   |    1
2    |  #   |    1
...
255  |  �   |    1  # <---- single character tokens up to here
256  |   t  |    2
257  |   a  |    2
...
298  | ent  |    3
299  |   n  |    2
"""

- **Note:** Entries 256 and 257 are not single-character values but double-character values (a white space + a letter), which is a little shortcoming of the original GPT-2 BPE Tokenizer (this has been improved in the GPT-4 tokenizer)

### 1.2 Building the vocabulary

- The goal of BPE tokenization algorithm is to build a vocabulary of commonly occuring subwords like `298: ent` (which can be found in **entangle, entertain, enter, entrance, entity, ..., for example), or even complete words like:

In [8]:
for i in [318, 617, 1212, 2420]:
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i:<8} | {decoded:<8}")

318      |  is     
617      |  some   
1212     | This    
2420     |  text   


- The BPE algorithm was originally described in 1994: ["A New Algorithm for Data Compression"](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf) by Philip Gage
- Before we get to the actual code implementation, the form that is used for LLM tokenizers can be summarized as described in the following sections

### 1.3 BPE algorithm outline

**1. Identify frequent pairs**
- In each iteration, scan the text to find the most commonly occuring pair of bytes (or characters)

**2. Replace and record**
- Replace that pair with a new placeholder ID (one not already in use, e.g., if we start with 0...255, the first placeholder would be 256)
- Record this mapping in a lookup table
- The size of the lookup table is a hyperparameter, also called "vocabulary size" (for GPT-2, that's 50,257)

**3. Repeat until no gains**
- Keep repeating steps 1 and 2, continually merging the most frequent pairs
- Stop when no further compression is possible (e.g., no pair occurs more than once)

**Decompression (decoding)**
- To restore the original text, reverse the process by substituting each ID with its corresponding pair, using the lookup table

### 1.4 BPE algorithm example

#### 1.4.1 Concrete example of the encoding part (first three steps in section 1.3)
- Suppose we have the text (training dataset) `the cat in the hat` from which we want to build the vocabulary from a BPE tokenizer

**Iteration 1**
1. Identify frequent pairs
- In this text, "th" appears twice (at the beginning and before the second "e").
2. Replace and record
- Replace "th" with a new token ID that is not already in use, e.g. `256`.
- The new text is: `<256>e cat in <256>e hat`
- The new vocabulary is
```text
    0: ...
    ...
    256: "th"
```
**Iteration 2**
1. Identify frequent pairs
- In the text `<256>e cat in <256>e hat`, the pair `<256>e` appears twice.
2. Replace and record
- Replace `<256>e` with a new token ID that is not already in use, for example, `257`.
- The new text is: `<257> cat in <257> hat`
- The updated vocabulary is:
```text
    0: ...
    ...
    256: "th"
    257: "<256>e"
```
**Iteration 3**
1. Identify frequent pairs
- In the text `<257> cat in <257> hat`, the pair `<257> ` appears twice.
2. Replace and record
- Replace `<257> ` with a new token ID that is not already in use, for example, `258`.
- The new text is: `<258>cat in <258>hat`
- The updated vocabulary is:
```text
    0: ...
    ...
    256: "th"
    257: "<256>e"
    258: "<257> "
```
- Repeat until no gains or we reach vocabulary size

#### 1.4.2 Concrete example of the decoding part (last step in section 1.3)
- To restore the original text, we reverse the process by substituting each token ID with its corresponding pair in the reverse order they were introduced
- Start with the final compressed text: `<258>cat in <258>hat`
- Substitute `<258>` → `<257> `: `<257> cat in <257> hat`
- Substitute `<257>` → `<256>e`: `<256>e cat in <256>e hat`
- Substitute `<256>` → "th": `the cat in the hat`

## 2. A simple BPE implementation
- Below is an implementation of this algorithm described above as a Python class that mimics the `tiktoken` Python user interface
- **Note:** The encoding part above describes the original training step via `train()`; however, the `encode()` method works similarly (although it looks a bit more complicated because of the special token handling):

1. Split the input text into individual bytes
2. Repeatedly find & replace (merge) adjacent tokens (pairs) when they match any pair in the learned BPE merges (from highest to lowest "rank," i.e., in the order they were learned)
3. Continue merging until no more merges can be applied
4. The final list of token IDs is the encoded output

In [ ]:
import json
from typing import Literal, Optional
from collections import Counter, deque
import re

class BPETokenizerSimple:
    def __init__(self) -> None:
        # Maps token_id to token_str
        self.vocab = {}
        # Maps token_str to token_id
        self.inverse_vocab = {}
        # Dictionary of BPE merges: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

        # For the official OpenAI GPT-2 merges, use a rank dict:
        #   of form {(string_A, string_B): rank}, where lower rank = higher priority
        self.bpe_ranks = {}
    
    def train(self, text: str, vocab_size: int, allowed_special: set = {"<|endoftext|>"}) -> None:
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Pre-tokenize training text using the same boundary rules as encode()
        tokens = self.pretokenize_text(text)

        # Initialize vocab with unique characters, including "Ġ" if present
        # Start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]
        # Extend unique_chars with characters from tokens that are not already included
        unique_chars.extend(char for char in sorted({char for token in tokens for char in token}) if char not in unique_chars)
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")
        
        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id
        
        # Tokenize each pre-token into character IDs
        token_id_sequences = [
            [self.inverse_vocab[char] for char in token]
            for token in tokens
        ]

        # BPE steps 1-3: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_id_sequences, mode="most")
            if pair_id is None: # No more pairs to merge. Stopping training.
                break

            token_id_sequences = self.replace_pair(token_id_sequences, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id
        
        # Build the vocabulary with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id
    
    def load_vocab_and_merges_from_openai(self, vocab_path: str, bpe_merges_path: str) -> None:
        """
        Load pre-trained vocabulary and BPE merges from OpenAI's GPT-2 files.

        Args:
            vocab_path (str): Path to the vocab file (GPT-2 calls it 'encoder.json').
            bpe_merges_path (str): Path to the bpe_merges file  (GPT-2 calls it 'vocab.bpe').
        """
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            # OpenAI's structure is {token_str: id}
            self.vocab = {int(v): k for k, v in loaded_vocab.items()}
            self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}
        
        # Must have GPT-2's printable newline character 'Ċ' (U+010A) at id 198
        if "Ċ" not in self.inverse_vocab or self.inverse_vocab["Ċ"] != 198:
            raise KeyError("Vocabulary missing GPT-2 newline glyph 'Ċ' at id 198.")
    
        # Must have <|endoftext|> at 50256
        if "<|endoftext|>" not in self.inverse_vocab or self.inverse_vocab["<|endoftext|>"] != 50256:
            raise KeyError("Vocabulary missing <|endoftext|> at id 50256.")

        # Provide a convenience alias for '\n' -> 198
        # Keep printable character 'Ċ' in vocab so BPE merges keep working
        if "\n" not in self.inverse_vocab:
            self.inverse_vocab["\n"] = self.inverse_vocab["Ċ"]

        if "\r" not in self.inverse_vocab:
            if 201 in self.vocab:
                self.inverse_vocab["\r"] = 201
            else:
                raise KeyError("Vocabulary missing carriage return token at id 201.")

        # Load GPT-2 merges and store ranks
        self.bpe_ranks = {}
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            lines = file.readlines()
            if lines and lines[0].startswith("#"):
                lines = lines[1:]
            
            rank = 0
            for line in lines:
                token1, *rest = line.strip().split()
                if len(rest) != 1:
                    continue
                token2 = rest[0]
                if token1 in self.inverse_vocab and token2 in self.inverse_vocab:
                    self.bpe_ranks[(token1, token2)] = rank
                    rank += 1
                else:
                    # Safe to skip pairs whose symbols are not in vocab
                    pass

    def encode(self, text: str, allowed_special: Optional[set] = None) -> list[int]:
        """
        Encode the input text into a list of token IDs, with tiktoken-style handling of special tokens.
    
        Args:
            text (str): The input text to encode.
            allowed_special (set or None): Special tokens to allow passthrough. If None, special handling is disabled.
    
        Returns:
            List of token IDs.
        """
        specials_in_vocab = [
            tok for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>")
        ]
        if allowed_special is None:
            disallowed = [tok for tok in specials_in_vocab if tok in text]
        else:
            disallowed = [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
        
        if disallowed:
            raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

        
        token_ids = []
        # If some specials are allowed, split around them and passthrough those ids
        if allowed_special is not None and len(allowed_special) > 0:
            special_pattern = "(" + "|".join(
                re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)
            ) + ")"

            last_index = 0
            for match in re.finditer(special_pattern, text):
                prefix = text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None)) # encode prefix normally

                special_token = match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token {special_token} not found in vocabulary.")
                last_index = match.end()
            
            text = text[last_index:] # remainder to process normally

            # Extra guard for any other special literals left over
            disallowed = [
                tok for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
            ]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

        tokens = self.pretokenize_text(text)

        for token in tokens:
            if token in self.inverse_vocab:
                # token is contained in the vocabulary as is
                token_ids.append(self.inverse_vocab[token])
            else:
                # Attempt to handle subword tokenization via BPE
                token_ids.extend(self.tokenize_with_bpe(token))
        
        return token_ids
    
    def tokenize_with_bpe(self, token: str) -> list[int]:
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Tokenize the token into individual characters (as initial token IDs)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")
        # If we haven't loaded OpenAI's GPT-2 merges
        if not self.bpe_ranks:
            can_merge = True
            while can_merge and len(token_ids) > 1:
                can_merge = False
                new_tokens = []
                i = 0
                while i < len(token_ids) - 1:
                    pair = (token_ids[i], token_ids[i + 1])
                    if pair in self.bpe_merges:
                        merged_token_id = self.bpe_merges[pair]
                        new_tokens.append(merged_token_id)
                        # Uncomment for educational purposes:
                        # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]})")
                        i += 2 # Skip the next token as it's merged
                        can_merge = True
                    else:
                        new_tokens.append(token_ids[i])
                        i += 1
                if i < len(token_ids):
                    new_tokens.append(token_ids[i])
                token_ids = new_tokens
            return token_ids
        
        # Otherwise, do GPT-2-style merging with the ranks:
        # 1) Convert token_ids back to string "symbols" for each ID
        symbols = [self.vocab[id_num] for id_num in token_ids]

        # Repeatedly merge all occurrences of the lowest-rank pair
        while True:
            # Collect all adjacent pairs
            pairs = set(zip(symbols, symbols[1:]))
            if not pairs:
                break

            # Find the pair with the best (lowest) rank
            min_rank = float("inf")
            bigram = None
            for p in pairs:
                r = self.bpe_ranks.get(p, float("inf"))
                if r < min_rank:
                    min_rank = r
                    bigram = p
            
            # If no valid ranked pair is present, we're done
            if bigram is None or bigram not in self.bpe_ranks:
                break

            # Merge all occurrences of that pair
            first, second = bigram
            new_symbols = []
            i = 0
            while i < len(symbols):
                # If we see (first, second) at position i, merge them
                if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
                    new_symbols.append(first + second) # merged symbol
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols

            if len(symbols) == 1:
                break
        
        # Finally, convert merged symbols back to IDs
        merged_ids = [self.inverse_vocab[sym] for sym in symbols]
        return merged_ids
    
    def decode(self, token_ids: list[int]) -> str:
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        out = []
        for tid in token_ids:
            if tid not in self.vocab:
                raise ValueError(f"Token ID {tid} not found in vocab.")
            tok = self.vocab[tid]

            # Map GPT-2 special chars back to real chars
            if tid == 198 or tok == "\n":
                out.append("\n")
            elif tid == 201 or tok == "\r":
                out.append("\r")
            elif tok.startswith("Ġ"):
                out.append(" " + tok[1:])
            else:
                out.append(tok)
        return "".join(out)
    
    def save_vocab_and_merges(self, vocab_path: str, bpe_merges_path: str) -> None:
        """
        Save the vocabulary and BPE merges to JSON files.

        Args:
            vocab_path (str): Path to save the vocabulary.
            bpe_merges_path (str): Path to save the BPE merges.
        """
        # Save vocabulary
        with open(vocab_path, "w", encoding="utf-8") as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)
        
        # Save BPE merges as a list of dictionaries
        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id} for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)
    
    def load_vocab_and_merges(self, vocab_path: str, bpe_merges_path: str) -> None:
        """
        Load the vocabulary and BPE merges from JSON files.

        Args:
            vocab_path (str): Path to the vocabulary file.
            bpe_merges_path (str): Path to the BPE merges file.
        """
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(k): v for k, v in loaded_vocab.items()}
            self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}
        
        # Load BPE merges
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            merges_list = json.load(file)
            for merge in merges_list:
                pair = tuple(merge["pair"])
                new_id = merge["new_id"]
                self.bpe_merges[pair] = new_id
    
    @staticmethod
    def pretokenize_text(text: str) -> list[str]:
        tokens = []
        parts = re.split(r'(\r\n|\r|\n)', text)
        for part in parts:
            if part == "":
                continue
            if part == "\r\n":
                tokens.append("\r")
                tokens.append("\n")
                continue
            if part == "\r":
                tokens.append("\r")
                continue
            if part == "\n":
                tokens.append("\n")
                continue

            # Normal chunk without line breaks:
            # - If spaces precede a word, prefix the first word with 'Ġ' and
            #   add standalone 'Ġ' for additional spaces
            # - If spaces trail the chunk (e.g., before a newline) add
            #   standalone 'Ġ' tokens (tiktoken produces id 220 for 'Ġ')
            pending_spaces = 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces += len(m.group(1))
                else:
                    word = m.group(2)
                    if pending_spaces > 0:
                        for _ in range(pending_spaces - 1):
                            tokens.append("Ġ")  # remaining spaces as standalone
                        tokens.append("Ġ" + word)   # one leading space
                        pending_spaces = 0
                    else:
                        tokens.append(word)
            # Trailing spaces (no following word): add standalone 'Ġ' tokens
            for _ in range(pending_spaces):
                tokens.append("Ġ")
        return tokens

    
    @staticmethod
    def find_freq_pair(token_id_sequences: list[list[int]], mode: Literal["most", "least"] = "most") -> tuple[int, int]:
        pairs = Counter(pair for token_ids in token_id_sequences for pair in zip(token_ids, token_ids[1:]))

        if not pairs:
            return None
        
        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")
    
    @staticmethod
    def replace_pair(token_id_sequences: list[list[int]], pair_id: tuple[int, int], new_id: int) -> list[list[int]]:
        replaced_sequences = []
        for token_ids in token_id_sequences:
            dq = deque(token_ids)
            replaced = []

            while dq:
                current = dq.popleft()
                if dq and (current, dq[0]) == pair_id:
                    replaced.append(new_id)
                    # Remove the 2nd token of the pair, 1st was already removed
                    dq.popleft()
                else:
                    replaced.append(current)
            
            replaced_sequences.append(replaced)
        
        return replaced_sequences

### Edge-case handling for short token sequences

BPE merges require adjacent token pairs.
If the token sequence has fewer than 2 items, no pair exists, so `find_freq_pair` returns `None` and training stops gracefully.

In [10]:
tok = BPETokenizerSimple()

assert tok.find_freq_pair([]) is None
assert tok.find_freq_pair([[42]]) is None

tok.train("", vocab_size=270)
tok.train("H", vocab_size=270)
tok.train("He", vocab_size=270)

print("Edge-case checks passed.")

Edge-case checks passed.


## 3. BPE implementation walkthrough

- In practice, it is recommended to use tiktoken library as implementation above focuses on readability and educational purposes, not on performance
- The usage is similar to tiktoken, except that tiktoken does not have training method
- Let's see how `BPETokenizerSimple` works by going through some examples below

### Utils

In [13]:
import os
import requests

def download_file_if_absent(url, filename, search_dirs):
    for directory in search_dirs:
        file_path = os.path.join(directory, filename)
        if os.path.exists(file_path):
            print(f"{filename} already exists in {file_path}")
            return file_path

    target_path = os.path.join(search_dirs[0], filename)
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(target_path, "wb") as out_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    out_file.write(chunk)
        print(f"Downloaded {filename} to {target_path}")
    except Exception as e:
        print(f"Failed to download {filename}. Error: {e}")

    return target_path

### 3.1 Training, encoding and decoding

- Let's first load some text as our training dataset:

In [ ]:
verdict_path = download_file_if_absent(
    url=(
         "https://raw.githubusercontent.com/rasbt/"
         "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
         "the-verdict.txt"
    ),
    filename="the-verdict.txt",
    search_dirs=["../../ch02/01_main-chapter-code/", "../01_main-chapter-code/", "."]
)

with open(verdict_path, "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

the-verdict.txt already exists in ../01_main-chapter-code/the-verdict.txt


- Next, let's initialise and train the BPE tokenizer with vocabulary size of 1,000
- Note that the vocabulary is already 256 by default due to the byte values discussed earlier, so we are only "learning" 745 vocabulary entries
- For comparison, the GPT-2 vocabulary is 50,257 tokens, the GPT-4 vocabulary is 100,256 (`cl100k_base` in tiktoken), and GPT-4o uses 199,997 tokens (`o200k_base` in tiktoken); they have all much bigger training sets compared to our simple example text above

In [12]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

In [13]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- This vocabulary is created merging 742 times

```
= 1000 - len(range(0, 256)) - len(special_tokens) - "Ġ" = 1000 - 256 - 1 - 1 = 742
```

In [14]:
print(len(tokenizer.bpe_merges))

742


- This means that the first 256 entries are single-character tokens

- Next, let's use the created merges via the `encode` method to encode some text:

In [15]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[74, 361, 310, 109, 98, 420, 397, 100, 300, 428, 116, 121, 519, 699, 299, 808, 534]


In [16]:
input_text = "Jack embraced beauty through art and life.<|endoftext|> "
token_ids = tokenizer.encode(input_text, allowed_special={"<|endoftext|>"})
print(token_ids)

[74, 361, 310, 109, 98, 420, 397, 100, 300, 428, 116, 121, 519, 699, 299, 808, 534, 257, 256]


In [17]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 56
Number of token IDs: 19


- From the lengths above, we can see that a 42-character sentence was encoded into 17 token IDs, effectively cutting the input length roughly in half compared to a character-byte-based encoding

- **Note:** The vocabulary itself is used in the `decode()` method, which allows us to map the token IDs back into text:

In [18]:
print(token_ids)

[74, 361, 310, 109, 98, 420, 397, 100, 300, 428, 116, 121, 519, 699, 299, 808, 534, 257, 256]


In [19]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.<|endoftext|> 


- Iterating over each token ID can give us a better understanding of how the token IDs are decoded via vocabulary:

In [20]:
for token_id in token_ids:
    print(f"{token_id:<6} -> {tokenizer.decode([token_id]):<6}")

74     -> J     
361    -> ack   
310    ->  e    
109    -> m     
98     -> b     
420    -> ra    
397    -> ce    
100    -> d     
300    ->  be   
428    -> au    
116    -> t     
121    -> y     
519    ->  through
699    ->  art  
299    ->  and  
808    ->  lif  
534    -> e.    
257    -> <|endoftext|>
256    ->       


- As we can see, most token IDs represent 2-character subwords; that's because the training data text is very short with not that many repetitive words, and because we used a relatively small vocabulary size

- As a summary, calling `decode(encode())` should be able to reproduce arbitrary input texts:

In [21]:
tokenizer.decode(tokenizer.encode("This is some text."))

'This is some text.'

In [22]:
tokenizer.decode(
    tokenizer.encode("This is some text with \n newline characters.")
)

'This is some text with \n newline characters.'

### 3.2 Saving and loading the tokenizer

- Next, let's look at how we can save the trained tokenizer for later use:

In [23]:
# Save trained tokenizer
tokenizer.save_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

In [24]:
# Load tokenizer
tokenizer2 = BPETokenizerSimple()
tokenizer2.load_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

- The loaded tokenizer should be able to produce the same results as before:

In [25]:
print(tokenizer2.decode(token_ids))

Jack embraced beauty through art and life.<|endoftext|> 


In [26]:
tokenizer2.decode(
    tokenizer2.encode("This is some text with \n newline characters.")
)

'This is some text with \n newline characters.'

### 3.3 Loading the original GPT-2 BPE tokenizer from OpenAI

In [17]:
# Download files if not already present in this directory

# Define the directories to search and the files to download
search_directories = ["../../ch02/02_bonus_bytepair-encoder/gpt2_model/", "../02_bonus_bytepair-encoder/gpt2_model/", "."]

files_to_download = {
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe": "vocab.bpe",
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json": "encoder.json"
}

# Ensure directories exist and download files if needed
paths = {}
for url, filename in files_to_download.items():
    paths[filename] = download_file_if_absent(url, filename, search_directories)

Downloaded vocab.bpe to ../../ch02/02_bonus_bytepair-encoder/gpt2_model/vocab.bpe
Downloaded encoder.json to ../../ch02/02_bonus_bytepair-encoder/gpt2_model/encoder.json


- Next, we load the files via the load_vocab_and_merges_from_openai method:

In [19]:
tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=paths["encoder.json"], bpe_merges_path=paths["vocab.bpe"]
)

- The vocabulary size should be `50257` as we can confirm via the code below:

In [20]:
len(tokenizer_gpt2.vocab)

50257

- We can now use the GPT-2 tokenizer via our `BPETokenizerSimple` object:

In [21]:
input_text = "This is some text"
token_ids = tokenizer_gpt2.encode(input_text)
print(token_ids)

[1212, 318, 617, 2420]


In [22]:
print(tokenizer_gpt2.decode(token_ids))

This is some text


- You can double-check that this produces the correct tokens using the tiktoken library:

In [23]:
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")

[1212, 318, 617, 2420]